In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline 

from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split

In [9]:
data=pd.read_csv("Datasets Primer Parcial/4-Bike Sharing/day.csv")
df = pd.read_csv('Datasets Primer Parcial/4-Bike Sharing/day.csv')
data.head()
#predecir demanda demanda de bicicletas Mediante Regresion Lineal
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 731 entries, 0 to 730
Data columns (total 16 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   instant     731 non-null    int64  
 1   dteday      731 non-null    str    
 2   season      731 non-null    int64  
 3   yr          731 non-null    int64  
 4   mnth        731 non-null    int64  
 5   holiday     731 non-null    int64  
 6   weekday     731 non-null    int64  
 7   workingday  731 non-null    int64  
 8   weathersit  731 non-null    int64  
 9   temp        731 non-null    float64
 10  atemp       731 non-null    float64
 11  hum         731 non-null    float64
 12  windspeed   731 non-null    float64
 13  casual      731 non-null    int64  
 14  registered  731 non-null    int64  
 15  cnt         731 non-null    int64  
dtypes: float64(4), int64(11), str(1)
memory usage: 91.5 KB


data=pd.read_csv("Primer parcial/Datasets Primer Parcial/4-Bike Sharing/bike-sharing-demand/train.csv")
data.head()
#predecir demanda demanda de bicicletas Mediante Regresion Lineal
data.info()nsforma el día de la semana (0-6) en columnas independientes para que el modelo aprenda, por ejemplo, que los domingos se alquilan menos bicis que los lunes.
mnth	Los meses (1-12) también se benefician de esto para capturar la estacionalidad de forma más precisa

In [10]:
"""
def one_hot(df, column):
    #Codificación one-hot manual para variables categóricas.
    uniques = sorted(df[column].unique())
    for val in uniques[1:]:  # drop_first=True
        df[f"{column}_{val}"] = (df[column] == val).astype(float)
    df.drop(column, axis=1, inplace=True)
    return df
"""

'\ndef one_hot(df, column):\n    #Codificación one-hot manual para variables categóricas.\n    uniques = sorted(df[column].unique())\n    for val in uniques[1:]:  # drop_first=True\n        df[f"{column}_{val}"] = (df[column] == val).astype(float)\n    df.drop(column, axis=1, inplace=True)\n    return df\n'

Usamos Encoder para usar el oneHot de sklearn

In [11]:
encoder = OneHotEncoder(drop='first', sparse_output=False)
columnasCategoricas = ['season', 'yr', 'mnth', 'holiday', 'weekday']

datasetCodificado = encoder.fit_transform(data[columnasCategoricas])

print(datasetCodificado)  # Mostrar el array
print(datasetCodificado.shape)

[[0. 0. 0. ... 0. 0. 1.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 1.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
(731, 22)


Normalizamos datos

In [12]:
def normalizarCaracteristicas(x):
    x_norm = x.copy()
    mu = np.zeros(x.shape[1])
    sigma = np.zeros(x.shape[1])
    
    mu = np.mean(x, axis=0)
    sigma = np.std(x, axis=0)
    x_norm = (x - mu) / sigma
    return x_norm,mu,sigma

In [13]:
# Extraer las columnas numéricas
caracteristicasNumericas = data[['temp', 'atemp', 'hum', 'windspeed']].values

# Normalizarlas
caracteristicasNormalizadas, mu, sigma = normalizarCaracteristicas(caracteristicasNumericas)

# Normalizar Y también
y_datos = data['cnt'].values.reshape(-1, 1)
y_normalizada, mu_y, sigma_y = normalizarCaracteristicas(y_datos)
y_normalizada = y_normalizada.flatten()  # Volver a 1D

# IMPORTANTE: Convertir a escalares
mu_y = mu_y[0]
sigma_y = sigma_y[0]

# Combinar X
X = np.hstack([datasetCodificado, caracteristicasNormalizadas])

# Dividir en train/test
X_train, X_test, y_train, y_test = train_test_split(X, y_normalizada, test_size=0.2, random_state=42)